# Wug paper figures & stimuli

Everything in this notebook uses **existing** results / data in the repo. Run it with the
`wug-test-interp` conda env kernel, from the repo root.

Contents:

1. PDF figure: column of singular wug images, column of plural wug images
2. LaTeX table of image-condition stimuli
3. LaTeX table of text (syntax) condition stimuli
4. LaTeX table of the nouns used to initialize the mean-embedding cluster
5. The full dev set (evaluation minimal pairs), as a table and a LaTeX `longtable`
6. Averaged training loss curves (text vs. vision) for the 2B and 4B models
7. Free-form generation from Qwen3-VL-2B with the learned syntax `[wug]`/`[wugs]` embeddings
8. The chat template, rendered on a real training example

In [ ]:
import os
import sys
import glob
import textwrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

REPO = os.path.abspath(".")
assert os.path.isdir(os.path.join(REPO, "core")), f"Run from repo root, got {REPO}"
if REPO not in sys.path:
    sys.path.insert(0, REPO)

CACHE_DIR = "/mnt/dv/wid/projects3/Rogers-muri-human-ai/zstuddiford"

# Data / results paths (all pre-existing in the repo)
IMAGE_DIR      = "data/embeddings/train/im/creature_1"
IMAGE_TRAIN_CSV = "data/embeddings/train/text/image_train_1.csv"
SYNTAX_TRAIN_CSV = "data/embeddings/train/text/syntax_train_1.csv"
NOUN_INIT_TXT  = "data/embeddings/init/noun_init.txt"
DEV_EVAL_CSV   = "data/embeddings/dev/dev_eval.csv"
CI_ROOT        = "results/train/CI_seed_runs"

FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)

## 1. Figure: singular vs. plural wug images

Two columns (singular on the left, plural on the right), one row per image index.
Saved as a vector PDF.

In [ ]:
singular_paths = sorted(glob.glob(os.path.join(IMAGE_DIR, "singular*.png")))
plural_paths   = sorted(glob.glob(os.path.join(IMAGE_DIR, "plural*.png")))
print(f"{len(singular_paths)} singular, {len(plural_paths)} plural images from {IMAGE_DIR}")

n_rows = max(len(singular_paths), len(plural_paths))
fig, axes = plt.subplots(n_rows, 2, figsize=(4.0, 2.0 * n_rows))
axes = np.atleast_2d(axes)

for col, (paths, title) in enumerate([(singular_paths, "singular"), (plural_paths, "plural")]):
    for row in range(n_rows):
        ax = axes[row, col]
        ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
        if row < len(paths):
            ax.imshow(Image.open(paths[row]).convert("RGB"))
        else:
            ax.axis("off")
        if row == 0:
            ax.set_title(f"[wug] ({title})" if col == 0 else f"[wugs] ({title})", fontsize=11)

fig.subplots_adjust(wspace=0.02, hspace=0.02)
out_pdf = os.path.join(FIG_DIR, "wug_stimuli_images.pdf")
fig.savefig(out_pdf, bbox_inches="tight", dpi=300)
print("wrote", out_pdf)
plt.show()

## 2–3. LaTeX tables of the training stimuli

`escape_latex` handles the `[wug]` brackets and any stray specials; the two conditions share
one formatter so the tables come out identical in structure.

In [ ]:
def escape_latex(s):
    """Escape LaTeX specials and typeset [wug]/[wugs] literally."""
    repl = {
        "\\": r"\textbackslash{}", "&": r"\&", "%": r"\%", "$": r"\$",
        "#": r"\#", "_": r"\_", "{": r"\{", "}": r"\}",
        "~": r"\textasciitilde{}", "^": r"\textasciicircum{}",
    }
    out = "".join(repl.get(ch, ch) for ch in str(s))
    return out.replace("[", "{[}").replace("]", "{]}")


def stimuli_table(csv_path, caption, label):
    """Side-by-side singular/plural stimuli table from a training CSV."""
    df = pd.read_csv(csv_path)
    df["type"] = df["type"].astype(str).str.strip().str.lower()
    df["sentence"] = df["sentence"].astype(str).str.strip()
    sing = df.loc[df["type"] == "singular", "sentence"].tolist()
    plur = df.loc[df["type"] == "plural", "sentence"].tolist()
    n = max(len(sing), len(plur))
    sing += [""] * (n - len(sing))
    plur += [""] * (n - len(plur))

    lines = [
        r"\begin{table}[t]",
        r"\centering",
        r"\small",
        r"\begin{tabular}{@{}p{0.44\linewidth}p{0.44\linewidth}@{}}",
        r"\toprule",
        r"Singular ([wug]) & Plural ([wugs]) \\".replace("[", "{[}").replace("]", "{]}"),
        r"\midrule",
    ]
    for s, p in zip(sing, plur):
        lines.append(f"{escape_latex(s)} & {escape_latex(p)} \\\\")
    lines += [
        r"\bottomrule",
        r"\end{tabular}",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        r"\end{table}",
    ]
    return "\n".join(lines)

### 2. Image-condition stimuli

In [ ]:
print(stimuli_table(
    IMAGE_TRAIN_CSV,
    caption="Training stimuli for the image condition. Each sentence is paired with a "
            "singular or plural creature image.",
    label="tab:stimuli-image",
))

### 3. Text (syntax) condition stimuli

In [ ]:
print(stimuli_table(
    SYNTAX_TRAIN_CSV,
    caption="Training stimuli for the text (syntax) condition. No image is shown; the novel "
            "token must be learned from syntactic context alone.",
    label="tab:stimuli-syntax",
))

## 4. Nouns used to initialize the embedding cluster

`[wug]`/`[wugs]` are initialized near the mean of these nouns' embeddings, with noise scaled
to the mean singular–plural distance (see `core/train/embed_train.py --embed_init`).

In [ ]:
with open(NOUN_INIT_TXT) as f:
    init_words = [w.strip() for w in f if w.strip()]

print(f"{len(init_words)} initialization words\n")
print(", ".join(init_words))
print()

N_COLS = 6
rows = [init_words[i:i + N_COLS] for i in range(0, len(init_words), N_COLS)]
lines = [
    r"\begin{table}[t]",
    r"\centering",
    r"\small",
    r"\begin{tabular}{@{}" + "l" * N_COLS + r"@{}}",
    r"\toprule",
]
for row in rows:
    row = row + [""] * (N_COLS - len(row))
    lines.append(" & ".join(escape_latex(w) for w in row) + r" \\")
lines += [
    r"\bottomrule",
    r"\end{tabular}",
    r"\caption{The " + str(len(init_words)) + r" nouns whose mean embedding defines the "
    r"initialization cluster for the novel tokens {[}wug{]} and {[}wugs{]}.}",
    r"\label{tab:noun-init}",
    r"\end{table}",
]
print("\n".join(lines))

## 5. The dev set (evaluation minimal pairs)

`data/embeddings/dev/dev_eval.csv` is the minimal-pair set the agreement eval scores: each row
is a grammatical `good` sentence and its ungrammatical `bad` counterpart, and accuracy is the
fraction of pairs where the model gives `good` the higher sequence score. Items are labelled
singular vs. plural by which novel token appears in the `good` sentence, matching the
`singular_eval_idx` / `plural_eval_idx` split in `core/train/embed_train.py`.

Printed twice: a readable pandas table, then a LaTeX `longtable` (a few hundred rows, so it has
to break across pages -- needs `\usepackage{longtable,booktabs}` in the preamble).

In [ ]:
dev = pd.read_csv(DEV_EVAL_CSV)
dev["good"] = dev["good"].astype(str).str.strip()
dev["bad"]  = dev["bad"].astype(str).str.strip()
dev["type"] = np.where(dev["good"].str.contains(r"\[wug\]", regex=True), "singular", "plural")

print(f"{len(dev)} minimal pairs from {DEV_EVAL_CSV}")
print(dev["type"].value_counts().to_string())
print()

with pd.option_context("display.max_rows", None, "display.max_colwidth", 80,
                       "display.width", 220):
    print(dev[["type", "good", "bad"]].to_string(index=True))

In [ ]:
CAPTION = (r"The full dev set: " + str(len(dev)) + r" grammatical/ungrammatical minimal pairs "
           r"used to evaluate the learned {[}wug{]}/{[}wugs{]} embeddings.")
HEADER = r"\# & Type & Grammatical (good) & Ungrammatical (bad) \\"

lines = [
    r"\begin{longtable}{@{}rlp{0.38\linewidth}p{0.38\linewidth}@{}}",
    r"\caption{" + CAPTION + r"}\label{tab:dev-set}\\",
    r"\toprule",
    HEADER,
    r"\midrule",
    r"\endfirsthead",
    r"\multicolumn{4}{@{}l}{\small\itshape (continued from previous page)}\\",
    r"\toprule",
    HEADER,
    r"\midrule",
    r"\endhead",
    r"\midrule",
    r"\multicolumn{4}{r@{}}{\small\itshape (continued on next page)}\\",
    r"\endfoot",
    r"\bottomrule",
    r"\endlastfoot",
]
for i, row in enumerate(dev.itertuples(index=False), start=1):
    lines.append(f"{i} & {row.type} & {escape_latex(row.good)} & {escape_latex(row.bad)} \\\\")
lines.append(r"\end{longtable}")

print("\n".join(lines))

## 6. Averaged loss curves (existing CI seed runs)

Reads `epoch_stats.csv` from every seed under `results/train/CI_seed_runs/`. Runs early-stop at
different epochs, so each epoch is averaged over whatever seeds are still alive there; the
shaded band is a 95% CI over seeds.

**Note on `avg_ce`:** the image-condition CI runs were written by an older version of
`embed_train.py` and their `epoch_stats.csv` has an *empty* `avg_ce` column (`sing_ce`/`plur_ce`
are fine). Plotting `avg_ce` directly therefore silently drops the vision curves. The loader
below falls back to `step_stats.csv`, where per-step `ce_loss` averaged within an epoch
reproduces `avg_ce` exactly on runs that have both — so text and vision stay comparable.

In [ ]:
CI_RUNS = {
    ("2B", "text"):   f"{CI_ROOT}/lr_ci_results_Qwen3-VL-2B-Instruct_text_lr0p001_50seeds",
    ("2B", "vision"): f"{CI_ROOT}/lr_ci_results_Qwen3-VL-2B-Instruct_image_lr0p001_50seeds",
    ("4B", "text"):   f"{CI_ROOT}/lr_ci_results_Qwen3-VL-4B-Instruct_text_lr0p001_50seeds",
    ("4B", "vision"): f"{CI_ROOT}/lr_ci_results_Qwen3-VL-4B-Instruct_image_lr0p001_50seeds",
}


def load_epoch_stats(run_dir):
    """Stack per-epoch stats across all seeds in a CI run dir.

    Returns a frame with columns [seed, epoch, avg_ce, sing_ce, plur_ce, ...]. Where
    epoch_stats.csv has no usable avg_ce (the image runs), it is rebuilt as the mean
    per-step ce_loss within each epoch from step_stats.csv.
    """
    paths = sorted(glob.glob(os.path.join(run_dir, "seed_*", "*", "epoch_stats.csv")))
    frames, n_rebuilt = [], 0
    for p in paths:
        seed = os.path.basename(os.path.dirname(os.path.dirname(p))).replace("seed_", "")
        df = pd.read_csv(p)
        df["seed"] = seed

        if "avg_ce" not in df.columns or df["avg_ce"].isna().all():
            step_path = os.path.join(os.path.dirname(p), "step_stats.csv")
            if os.path.exists(step_path):
                rebuilt = pd.read_csv(step_path).groupby("epoch")["ce_loss"].mean()
                df["avg_ce"] = df["epoch"].map(rebuilt)
                n_rebuilt += 1
            else:
                print(f"  !! {p}: no avg_ce and no step_stats.csv to rebuild from")
        frames.append(df)

    if not frames:
        print(f"  !! no epoch_stats.csv under {run_dir}")
        return None

    out = pd.concat(frames, ignore_index=True)
    note = f", avg_ce rebuilt from step_stats for {n_rebuilt}" if n_rebuilt else ""
    print(f"  {os.path.basename(run_dir)}: {len(paths)} seeds, "
          f"max epoch {out['epoch'].max()}{note}")
    return out


stats = {}
for key, d in CI_RUNS.items():
    stats[key] = load_epoch_stats(d)

# Sanity check: nothing should be silently all-NaN in the column we plot.
for key, df in stats.items():
    if df is not None:
        frac = df["avg_ce"].notna().mean()
        print(f"{key}: avg_ce non-null {frac:.1%}")
        assert frac > 0, f"{key} has no usable avg_ce"

In [ ]:
LOSS_COL = "avg_ce"   # per-epoch mean cross-entropy on the novel-token positions
COLORS = {"text": "#1f77b4", "vision": "#d62728"}

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.6), sharey=True)

for ax, model in zip(axes, ["2B", "4B"]):
    for cond in ["text", "vision"]:
        df = stats.get((model, cond))
        if df is None:
            continue
        df = df.dropna(subset=[LOSS_COL])
        if df.empty:
            print(f"!! no {LOSS_COL} data for {model}/{cond} -- nothing plotted")
            continue
        g = df.groupby("epoch")[LOSS_COL]
        mean, sd, n = g.mean(), g.std(ddof=1), g.count()
        ci = 1.96 * sd / np.sqrt(n.clip(lower=1))
        ax.plot(mean.index, mean.values, color=COLORS[cond], lw=1.8,
                label=f"{cond} (n={int(n.max())} seeds)")
        ax.fill_between(mean.index, mean - ci, mean + ci, color=COLORS[cond], alpha=0.20, lw=0)
    ax.set_title(f"Qwen3-VL-{model}")
    ax.set_xlabel("epoch")
    ax.legend(frameon=False, fontsize=9)
    ax.spines[["top", "right"]].set_visible(False)

axes[0].set_ylabel("mean CE on novel token")
fig.tight_layout()
out_pdf = os.path.join(FIG_DIR, "loss_curves_text_vs_vision.pdf")
fig.savefig(out_pdf, bbox_inches="tight")
print("wrote", out_pdf)
plt.show()

## 7. Free-form generation with the learned syntax embeddings (Qwen3-VL-2B)

Loading mirrors `core/interp/intervention.py::_inject_embeddings`:

1. build the `VLMScorer` with `cache_dir=CACHE_DIR`,
2. add the `" [wug]"` / `" [wugs]"` tokens and resize the embedding matrix,
3. write the saved `wug_embedding` / `wugs_embedding` rows into `embed_tokens`.

Embeddings and `lm_head` are tied, so the injected rows are both readable and generatable.

In [ ]:
import torch
from minicons import scorer
from utils.chat_templates import train_chat_template_noimage

MODEL_PATH      = "Qwen/Qwen3-VL-2B-Instruct"
EMBEDDINGS_PATH = "embeddings/Qwen3-VL-2B/syntax/qwen3_vl_2b_syntax.pt"
ADDED_TOKENS    = [" [wug]", " [wugs]"]

device = "cuda" if torch.cuda.is_available() else "cpu"
lm = scorer.VLMScorer(MODEL_PATH, device=device, torch_dtype=torch.bfloat16,
                      cache_dir=CACHE_DIR)

tok = lm.tokenizer.tokenizer          # inner tokenizer; lm.tokenizer is the processor
model = lm.model

to_add = [t for t in ADDED_TOKENS if t not in tok.get_vocab()]
if to_add:
    tok.add_tokens(to_add)
    old_len = model.resize_token_embeddings().weight.shape[0]
    model.resize_token_embeddings(old_len + len(to_add))

emb = model.model.language_model.embed_tokens
wug_id, wugs_id = [tok(t, add_special_tokens=False).input_ids[0] for t in ADDED_TOKENS]

rec = torch.load(EMBEDDINGS_PATH, map_location="cpu")
with torch.no_grad():
    emb.weight.data[wug_id]  = rec["wug_embedding"].to(emb.weight.device, dtype=emb.weight.dtype)
    emb.weight.data[wugs_id] = rec["wugs_embedding"].to(emb.weight.device, dtype=emb.weight.dtype)

print(f"injected {EMBEDDINGS_PATH}")
print(f"  saved model  : {rec.get('model_name')}   epoch: {rec.get('saved_epoch')}")
print(f"  ids in file  : wug={rec['wug_id']} wugs={rec['wugs_id']}")
print(f"  ids here     : wug={wug_id} wugs={wugs_id}")
assert (rec["wug_id"], rec["wugs_id"]) == (wug_id, wugs_id), \
    "Token id mismatch between checkpoint and current tokenizer"
print(f"  tied lm_head : {emb.weight.data_ptr() == model.lm_head.weight.data_ptr()}")

In [ ]:
PROMPTS = [
    "One [wug] was playing and another came to join it. Now there are",
    "I saw a single [wug] yesterday. Today I saw three",
    "There is one [wug] on the left and two",
    "Every [wug] in the field looked up. All of the",
    "The [wugs] were resting, but only one",
]


@torch.no_grad()
def generate(sentence, max_new_tokens=10, do_sample=False):
    """Greedy continuation, using the same user turn the model was trained with."""
    context = [
        {"role": "user", "content": [{"type": "text", "text": "Complete the sentence."}]},
        {"role": "assistant", "content": [{"type": "text", "text": sentence}]},
    ]
    prompt = lm.tokenizer.apply_chat_template(context, continue_final_message=True)
    enc = lm.tokenizer(text=[prompt], return_tensors="pt", padding=True)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=do_sample,
                         pad_token_id=tok.pad_token_id or tok.eos_token_id)
    new_ids = out[0, enc["input_ids"].shape[1]:]
    return tok.decode(new_ids, skip_special_tokens=True)


for sent in PROMPTS:
    cont = generate(sent)
    print(f"PROMPT     : {sent}")
    print(f"CONTINUATION: {cont!r}")
    print(f"FULL       : {sent}{cont}")
    print("-" * 78)

## 8. The chat template, rendered on a training example

`train_chat_template_noimage` is what the syntax condition trains on; the tokenization shows
exactly where the `[wug]` token lands.

In [ ]:
example = pd.read_csv(SYNTAX_TRAIN_CSV).iloc[0]["sentence"].strip()
rendered = train_chat_template_noimage(lm, example)

print("Training sentence:")
print(f"  {example!r}\n")
print("Rendered chat template (repr, so newlines/specials are visible):")
print(f"  {rendered!r}\n")
print("Rendered chat template (raw):")
print("-" * 78)
print(rendered)
print("-" * 78)

ids = lm.tokenizer(text=[rendered], return_tensors="pt")["input_ids"][0]
print(f"\n{len(ids)} tokens; [wug] id = {wug_id}, [wugs] id = {wugs_id}")
for i, t in enumerate(ids.tolist()):
    mark = "  <-- supervised novel token" if t in (wug_id, wugs_id) else ""
    print(f"  {i:3d}  {t:7d}  {tok.decode([t])!r}{mark}")